# LLM Fine-Tuning Deep Dive, Part 2 of 3: Parameter-Based Techniques + QLoRA & Quantization

> **This is Part 2 of a three-notebook fine-tuning arc:**
>
> 1. [Part 1: Data-based techniques](01-llm-finetuning-data-techniques.ipynb) -- continued
>    pretraining (full FT), SFT (LoRA), preference alignment / DPO (LoRA). Trains and saves three
>    checkpoints to `./checkpoints/` that this notebook reloads below.
> 2. **Part 2 (this notebook): Parameter-based techniques + QLoRA & quantization** -- how many/which
>    weights are updated (full fine-tuning, partial freezing, LoRA), then QLoRA (LoRA on a quantized
>    base) and a real, runnable look at post-training quantization for deployment.
> 3. [Part 3: Comparison & decision](03-llm-finetuning-comparison-and-decision.ipynb) -- head-to-head
>    evaluation of all six trained checkpoints, held-out perplexity, an ablation study, and the final
>    call on what Riverside House actually deploys.

**Recap of where Part 1 left off:** Riverside now has a model that knows its catalog (continued
pretraining with full FT), follows instructions (SFT with LoRA), and has attempted preference
alignment (DPO with LoRA, honestly reported as not-yet-converged with only 30 pairs). That's the
*data axis*. Part 1 made implicit parameter choices for each stage; this notebook makes the parameter
choice the *variable*: we use continued pretraining as the controlled test bed — same data throughout,
so any output difference comes only from how many weights updated. The most-used real-world pairings
(LoRA + SFT, LoRA + DPO) are already demonstrated in Part 1; the full 3 × 3 data × parameter matrix
appears in Part 3.

## Table of Contents (Part 2)

1. [Setup: Reloading Where Part 1 Left Off](#setup-reloading-where-part-1-left-off)
2. [Parameter-Based Axis: How Many Weights Do We Actually Update?](#parameter-based-axis-how-many-weights-do-we-actually-update)
   - [Concept 4: Full Fine-Tuning](#concept-4-parameter-based-full-fine-tuning)
   - [Concept 5: Partial Freezing](#concept-5-parameter-based-partial-fine-tuning-layer-freezing)
   - [Concept 6: LoRA](#concept-6-parameter-based-parameter-efficient-fine-tuning-lora)
   - [Concept 7: QLoRA](#concept-7-parameter-based-qlora--lora-on-a-quantized-base)
   - [Quantization for Deployment](#quantization-for-deployment-shrinking-the-model-riverside-actually-ships)
3. [Visual Comparison: Parameter Counts Across All Techniques](#visual-comparison-parameter-counts-across-all-techniques)

---

## Setup: Reloading Where Part 1 Left Off

Kernels don't share memory between notebooks. This section reloads everything needed from disk: the
tokenizer, corpus loader, `generate()` helper, baseline model, and the instruction-tuned LoRA adapter
from `./checkpoints/instruction-lora`.


In [ ]:
# Re-establishing Part 1's foundations -- see Part 1 for the reasoning behind each choice below.
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2-medium"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"
INSTRUCTION_PREFIX = "Continue the fiction narrative in the same style:\n\n"


def generate(model, prompt, max_new_tokens=60):
    """Generate a continuation, returning only the new tokens (full walkthrough is in Part 1)."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
        )
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    return completion if completion else "[model stopped immediately — sampled EOS as first token]"


print(f"Baseline completion (sanity check): {generate(base_model, PROMPT)}")


In [ ]:
# Corpus loader (identical to Part 1) + the tokenize_causal() helper Concept 5/6 reuse for training,
# plus every visualization/training/PEFT import this notebook needs.
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / "content"
if not CONTENT_DIR.exists():
    _fallback = Path.cwd() / "learning" / "genai" / "04-llm" / "content"
    if _fallback.exists():
        CONTENT_DIR = _fallback

print(f"Content directory: {CONTENT_DIR.absolute()}")

NOVELS = {
    "scifi":      "the-weight-of-distant-light",
    "fantasy":    "the-tidebound-accord",
    "mystery":    "the-cartographers-cipher",
    "historical": "the-silk-merchants-daughter",
    "cyberpunk":  "neural-drift",
    "horror":     "the-hollow-beneath",
    "literary":   "the-weight-of-tides",
}


def load_corpus_paragraphs(novels=None, max_chapters=10, min_len=200):
    """Load paragraphs from the multi-novel corpus for quick CPU demos (identical to Part 1)."""
    if novels is None:
        novels = list(NOVELS.keys())
    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            continue
        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            continue
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))[:max_chapters]
        for path in chapter_files:
            text = path.read_text(encoding="utf-8")
            for para in text.split("\n\n"):
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:
                    paragraphs.append(para)
    return paragraphs


def tokenize_causal(examples, tokenizer, max_length=128):
    """Standard next-token-prediction tokenization (identical to Part 1's Concept 1)."""
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length
    )
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
from datasets import Dataset
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")

print("Corpus loader, tokenize_causal(), and training/visualization imports ready.")


### Reloading Part 1's Instruction-Tuned LoRA Adapter

The "Cracking Open a Real LoRA-Adapted Model" subsection later in this notebook (Concept 6) inspects
a real, trained LoRA adapter -- specifically the instruction-tuning adapter from Part 1's Concept 2,
saved to `./checkpoints/instruction-lora`. Reloading it here needs its own fresh base-model instance
(the same "every PEFT wrapper gets its own base" pattern Part 1 used for `instruct_base` and
`lora_pt_base`), then `PeftModel.from_pretrained()` to attach the saved adapter weights.


In [ ]:
# Reload the instruction-tuned LoRA adapter Part 1 saved to disk (before DPO ran on top of it --
# this is the pre-DPO instruction-tuned checkpoint, exactly as advertised).
instruct_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = PeftModel.from_pretrained(
    instruct_base_reload, "./checkpoints/instruction-lora"
).to(device)
instruct_lora_model.eval()
print("Reloaded the instruction-tuned LoRA adapter from Part 1's checkpoint.")
print(generate(instruct_lora_model, INSTRUCTION_PREFIX + "Aria checked the Meridian and\n\n"))


---

## Parameter-Based Axis: How Many Weights Do We Actually Update?

**Riverside's question for this section:** IT has given us one laptop CPU and a deadline. We now have
a model that knows the lore, follows instructions, and matches editorial taste -- but which of the
three ways to _train_ it can we actually afford to run and re-run as the catalog grows?

### The Cost Problem

All three data-based techniques (continued pretraining, instruction tuning, preference alignment) work
by gradient descent on model weights. But **updating all weights is expensive:**

- **Memory:** ~355M parameters × (4 bytes per param + 8 bytes optimizer state) = ~4.3 GB just for
  `gpt2-medium`. For 70B models, this becomes **840 GB**.
- **Compute:** More trainable params = longer training time
- **Risk:** Full updates can "overwrite" the model's general knowledge (catastrophic forgetting) --
  which for Riverside means the assistant could start forgetting ordinary English while it's busy
  memorizing the sci-fi glossary.

### The Trade-Off Spectrum

Independent of _what data_ you train on, you can choose _how much of the model_ to update:

| Technique            | Trainable % | Memory  | Quality | Forgetting Risk | When to Use                         |
| -------------------- | ----------- | ------- | ------- | ---------------- | ------------------------------------ |
| **Full fine-tuning** | 100%        | Highest | Highest | Highest          | Small models, abundant compute      |
| **Partial freezing** | 10-30%      | Medium  | Medium  | Medium           | Limited compute budget              |
| **LoRA**             | <1%         | Lowest  | High    | Lowest           | Most production scenarios today     |
| **QLoRA**            | <1%         | Lowest of all (quantized base too) | High | Lowest | Largest models, tightest memory budget |

### The Two Axes Are Complementary, Not Either/Or

It's easy to read "data-based" (continued pretraining / instruction tuning / DPO) and "parameter-based"
(full FT / partial freezing / LoRA) as two competing menus you pick one item from. They aren't --
they're **independent choices that combine**: every stage of the data-based journey still needs an
answer to "how many weights move while learning this," so in practice you pick **one axis for what to
teach and one axis for how much of the model to touch**, and every one of the 9 combinations below is
a real, valid configuration with its own trade-off:

| Data objective ↓ / Parameter strategy → | **Full Fine-Tuning** (100%)                                                                     | **Partial Freezing** (~10-30%)                                                                | **LoRA** (<1%)                                                                                     |
| ---------------------------------------- | ------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------- |
| **Continued Pretraining**                | **Pros:** deepest vocabulary/style absorption. **Cons:** highest forgetting risk + cost -- overkill unless the domain shift is large and the model is small. | **Pros:** meaningful domain absorption at a fraction of the cost, low forgetting. **Cons:** still edits raw weights, can't be swapped out like an adapter. | **Pros:** cheapest and safest option, adapters are swappable per genre/domain. **Cons:** slightly lower ceiling for very large vocabulary shifts than full FT. |
| **Instruction Tuning (SFT)**             | **Pros:** maximum freedom to reshape behavior/format. **Cons:** expensive per iteration for a change that's usually about *format*, not deep knowledge -- rarely worth it. | **Pros:** middle ground, keeps most general ability while still learning the instruction format. **Cons:** manual layer-count tuning, not swappable between use cases. | **Pros:** the practical default (what Part 1 actually does) -- cheap to iterate on prompt templates, swappable per assistant persona. **Cons:** the adapter must be loaded at inference time. |
| **Preference Alignment (DPO)**           | **Pros:** theoretically most expressive. **Cons:** highest risk of all nine cells -- small preference datasets plus full-parameter updates invite overfitting/reward-hacking; rarely used in practice. | **Pros:** some safety margin vs. full FT. **Cons:** still more capacity than a narrow preference signal usually needs. | **Pros:** the practical default (also what Part 1 does) -- small, contained updates matched to how little preference data teams typically have. **Cons:** stacking adapters (SFT + DPO) adds bookkeeping if they aren't merged before deployment. |

The diagonal pattern above is exactly why production pipelines converge on **LoRA for every data-based
stage** once models get large: the parameter-efficient column is the safe default for instruction
tuning and DPO specifically because those stages usually have far less data than continued
pretraining, and full fine-tuning on a small preference dataset is a recipe for reward hacking, not
better alignment. Part 1 trained 5 of these 9 real combinations (continued pretraining × all
three parameter strategies, plus instruction tuning and DPO × LoRA) -- Part 3's **Technique
Combination Grid** revisits this exact matrix with real held-out perplexity numbers in each trained
cell, so the qualitative trade-offs above get a quantitative check.

The following cells apply all three strategies (plus QLoRA, further down) to the _same_
continued-pretraining objective, so parameter counts are directly comparable -- this is the data
Riverside's IT lead will actually want to see before signing off on a training budget.

---

### Concept 4 (Parameter-Based): Full Fine-Tuning

**What it is:** Every single weight in the model is unfrozen and updated by the optimizer.

**Pros:** The model has maximum "room" to adapt to Riverside's catalog.

**Cons:**

- Costs the most memory/compute
- Highest risk of catastrophic forgetting if the corpus is small (Riverside's 7 novels are tiny by
  LLM standards)
- For large models (>7B params), often infeasible without multi-GPU setups -- a non-starter for a
  publisher with one laptop

We already ran this in Part 1's Concept 1 (`./checkpoints/non-instruction-full`) -- that cell **is**
full fine-tuning. The cell below quantifies what "100% trainable" looks like for `gpt2-medium`.


In [ ]:
param_check_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
total = sum(p.numel() for p in param_check_model.parameters())
trainable = sum(p.numel() for p in param_check_model.parameters() if p.requires_grad)
print(
    f"Full fine-tuning: {trainable:,}/{total:,} parameters trainable ({trainable / total * 100:.1f}%)"
)
del param_check_model

### Concept 5 (Parameter-Based): Partial Fine-Tuning (Layer Freezing)

**Riverside's question for this section:** if full fine-tuning is the "maximum quality, maximum
laptop-fan-noise" option, is there a middle ground that still lets us re-train every time the catalog
grows, without waiting hours?

**The observation:** In transformer models, **early layers** learn general language features
(tokenization, basic syntax, common words) while **later layers** learn task-specific patterns. This
is similar to how early layers in CNNs detect edges, while later layers detect objects.

**The strategy:** Freeze everything, then selectively unfreeze:

- The last N transformer blocks (task-specific adaptation)
- The output head (final projection to vocabulary)

**Pros:**

- Much cheaper than full fine-tuning (only 10-30% of parameters)
- Less prone to catastrophic forgetting (general features preserved)
- No new architecture needed

**Cons:**

- Still edits raw model weights (can't easily "swap" like an adapter) -- so Riverside couldn't keep
  one editing-assistant checkpoint and one knowledge-base checkpoint without storing two full copies
  of `gpt2-medium`
- Choosing _how many_ layers to unfreeze is a manual hyperparameter
- Middle ground: not as cheap as LoRA, not as powerful as full fine-tuning

**Example:** For `gpt2-medium` (24 transformer blocks), we unfreeze only the last ~25% of blocks
(6 blocks) + output head.


### Visualizing Layer-by-Layer Freezing

Before we run the code, let's visualize **exactly which layers** in `gpt2-medium` (24 transformer
blocks) will be frozen vs. trainable when we unfreeze the last ~25% of blocks.

**The intuition:**

Think of the transformer as a **semantic refinement pipeline**:

| Layer                         | What it learns                                    | Freeze or Train? | Why?                                             |
| ----------------------------- | ------------------------------------------------- | ---------------- | ------------------------------------------------ |
| **Early blocks (~first 50%)** | Basic syntax, common words, tokenization patterns | FROZEN           | These are universal — no need to change          |
| **Middle blocks (~50-75%)**   | Mid-level semantics, phrase structure             | FROZEN           | Still mostly general-purpose                     |
| **Last ~25% of blocks**       | Task-specific patterns, domain adaptation         | TRAINABLE        | This is where domain/task specialization happens |
| **Output head**               | Final vocabulary distribution                     | TRAINABLE        | Must learn domain-specific words                 |

**Why this works:**

1. **Early layers = general features:** Just like CNNs learn edges in early layers, transformer early
   blocks learn general language structure that's useful for _any_ task.
2. **Late layers = task-specific:** The final blocks learn task/domain-specific patterns. By only
   training these, we adapt to our corpus without forgetting general English.
3. **Catastrophic forgetting prevention:** Freezing ~75% of the model preserves general language
   ability while allowing focused adaptation.


In [ ]:
# Visualize partial freezing: which layers are trainable?
from transformers import AutoConfig
from matplotlib.patches import Patch, Rectangle

_freeze_cfg = AutoConfig.from_pretrained(MODEL_NAME)
n_layers = _freeze_cfg.n_layer  # gpt2-medium has 24 transformer blocks
unfreeze_from = n_layers - max(2, n_layers // 4)  # unfreeze the last ~25% of blocks
layers = [f"Block {i}" for i in range(n_layers)] + ["Output Head"]
layer_positions = np.arange(len(layers))
tick_stride = max(1, n_layers // 12)  # keep y-axis labels readable regardless of depth

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(6, n_layers * 0.3)))

# Plot 1: Frozen vs Trainable blocks. With 24+ rows there isn't room for per-bar text labels
# without them overlapping, so color + a legend carries the FROZEN/TRAINABLE distinction instead.
colors = ["lightblue" if i < unfreeze_from else "coral" for i in range(n_layers)] + [
    "coral"
]
ax1.barh(
    layer_positions, [1] * len(layers), color=colors, edgecolor="black", linewidth=1.0
)

ax1.set_yticks(layer_positions[::tick_stride])
ax1.set_yticklabels([layers[i] for i in layer_positions[::tick_stride]])
ax1.set_xlim(0, 1)
ax1.set_xticks([])
ax1.set_title(
    f"Partial Fine-Tuning Strategy\n({MODEL_NAME}: {n_layers} blocks)",
    fontsize=12,
    fontweight="bold",
)
ax1.invert_yaxis()
ax1.legend(
    handles=[
        Patch(facecolor="lightblue", edgecolor="black", label="FROZEN"),
        Patch(facecolor="coral", edgecolor="black", label="TRAINABLE"),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
    fontsize=9,
)

# Plot 2: Gradient flow visualization
gradient_flow = (
    [0.0] * unfreeze_from
    + list(np.linspace(0.5, 1.0, n_layers - unfreeze_from))
    + [1.0]
)
ax2.barh(
    layer_positions,
    gradient_flow,
    color="green",
    alpha=0.7,
    edgecolor="black",
    linewidth=1.0,
    label="Relative gradient magnitude",
)
ax2.set_yticks(layer_positions[::tick_stride])
ax2.set_yticklabels([layers[i] for i in layer_positions[::tick_stride]])
ax2.set_xlabel("Gradient Magnitude (relative)", fontsize=10)
ax2.set_title("Gradient Flow During Backpropagation", fontsize=12, fontweight="bold")
ax2.invert_yaxis()
ax2.set_xlim(0, 1.1)

frozen_mid = unfreeze_from / 2
trainable_mid = unfreeze_from + (n_layers - unfreeze_from) / 2
ax2.text(
    0.05,
    frozen_mid,
    "No gradient\nflow",
    ha="center",
    va="center",
    fontsize=9,
    color="gray",
    fontweight="bold",
    style="italic",
)
ax2.text(
    0.75,
    trainable_mid,
    "Full gradient\nflow",
    ha="center",
    va="center",
    fontsize=9,
    color="darkgreen",
    fontweight="bold",
)
ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), fontsize=9)

plt.tight_layout()
plt.show()

# Calculate parameter breakdown
total_blocks = n_layers
frozen_blocks = unfreeze_from
trainable_blocks = total_blocks - frozen_blocks
frozen_pct = (frozen_blocks / total_blocks) * 100
trainable_pct = 100 - frozen_pct

print(f"\n{'=' * 70}")
print(f"Partial Fine-Tuning Configuration:")
print(f"{'=' * 70}")
print(f"  Total blocks:      {total_blocks}")
print(
    f"  Frozen blocks:     {frozen_blocks} (blocks 0-{frozen_blocks-1}) — {frozen_pct:.1f}%"
)
print(
    f"  Trainable blocks:  {trainable_blocks} (blocks {unfreeze_from}-{total_blocks-1}) — {trainable_pct:.1f}%"
)
print(f"  Output head:       TRAINABLE")
print(f"{'=' * 70}")
print(f"  Memory savings:    ~{frozen_pct:.0f}% less optimizer state")
print(f"  Forgetting risk:   LOW (general features preserved)")
print(f"  Adaptation power:  MEDIUM (targeted domain learning)")
print(f"{'=' * 70}")

In [ ]:
freeze_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

for param in freeze_model.parameters():
    param.requires_grad = False

### Selectively Unfreezing the Last ~25% of Blocks

Every parameter starts frozen above. This cell re-enables `requires_grad` on only the last few
transformer blocks (`unfreeze_from` onward) plus the final layer norm and output head -- the exact
split visualized earlier in this section -- so the optimizer only ever sees gradients for that
trainable slice.

In [ ]:
n_layers = freeze_model.config.n_layer  # gpt2-medium has 24 transformer blocks
unfreeze_from = n_layers - max(
    2, n_layers // 4
)  # unfreeze only the last ~25% of blocks

for name, param in freeze_model.named_parameters():
    if any(f"h.{i}." in name for i in range(unfreeze_from, n_layers)):
        param.requires_grad = True
    if "ln_f" in name or "lm_head" in name:
        param.requires_grad = True

trainable = sum(p.numel() for p in freeze_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in freeze_model.parameters())
print(
    f"Partial fine-tuning: {trainable:,}/{total:,} parameters trainable ({trainable / total * 100:.2f}%)"
)

### Building the Dataset and Running the Trainer

Same `tokenize_causal()`/`Trainer` pattern as continued pretraining, on a different 2-genre slice of
the corpus, with `learning_rate=1e-4` -- between full fine-tuning's `5e-5` and LoRA's `2e-4`, since
partial freezing updates more parameters than LoRA but far fewer than full fine-tuning.

In [ ]:
freeze_dataset = Dataset.from_dict(
    {"text": load_corpus_paragraphs(novels=["fantasy", "cyberpunk"], max_chapters=3)}
)
freeze_tokenized = freeze_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)

training_args_freeze = TrainingArguments(
    output_dir="./checkpoints/partial-freeze",
    per_device_train_batch_size=2,
    max_steps=60,  # gpt2-medium needs more steps than distilgpt2 did to actually settle
    logging_steps=10,
    save_strategy="no",
    learning_rate=1e-4,
    report_to="none",
)

trainer_freeze = Trainer(
    model=freeze_model, args=training_args_freeze, train_dataset=freeze_tokenized
)
trainer_freeze.train()
freeze_model.save_pretrained("./checkpoints/partial-freeze")
print("Saved partial (layer-freezing) fine-tune checkpoint.")

### Concept 6 (Parameter-Based): Parameter-Efficient Fine-Tuning (LoRA)

**Riverside's question for this section:** remember the two jobs -- an editing assistant _and_ a
knowledge base? With full fine-tuning or partial freezing, shipping both means storing two full
355M-parameter models. Is there a way to keep one frozen base model on disk and swap in a small,
task-specific adapter depending on who's asking?

**The key insight:** Instead of updating existing weights, **freeze the entire base model** and inject
small trainable matrices alongside key weight matrices.

**How LoRA works:**

For a weight matrix $W$ (e.g., attention projection), instead of updating $W \rightarrow W + \Delta W$,
we:

1. **Freeze** $W$ (no updates ever)
2. **Add** a low-rank decomposition: $\Delta W = BA$, computed as $B(Ax)$ -- $x$ hits $A$ first:
   - $A$ is $r \times d$ (rank-**reducing** projection: takes the $d$-dim input down to $r$ dims)
   - $B$ is $d \times r$ (rank-**expanding** projection: takes that $r$-dim result back up to $d$ dims)
   - $r \ll d$ (rank is much smaller than original dimension)

**Example:** For `gpt2-medium`'s attention (d=1024), with r=8:

- Original: 1024 × 1024 = **1,048,576 parameters**
- LoRA: (8×1024) + (1024×8) = **16,384 parameters** (1.56% of original)

**What to tune:**

- `r` (rank): Higher = more capacity but more parameters. Typical: 4-64.
- `target_modules`: Which weight matrices to adapt. For transformers: attention projections (`q`,
  `k`, `v`, `o`) and sometimes feed-forward layers.
- `lora_alpha`: Scaling factor (typical: 2×r)

**Pros:**

- **Tiny memory footprint:** Only adapter's optimizer state needed
- **Swappable:** Train multiple adapters on the same frozen base model, swap at inference -- this is
  the direct answer to Riverside's two-jobs problem: one frozen `gpt2-medium` on disk, an
  "editing-assistant" adapter and a "knowledge-base" adapter (a few MB each) swapped in per request
- **Mergeable:** Can merge $BA$ into $W$ for zero-latency deployment
- **Lowest forgetting risk:** Base weights never change

**Cons:**

- Slightly lower quality ceiling than full fine-tuning for extreme distribution shifts
- Adds hyperparameters to tune (`r`, `alpha`, `target_modules`)
- Inference needs adapter loaded/merged

**Related techniques:**

- **Adapters:** Small bottleneck layers inserted between transformer blocks (not demoed here)
- **Prefix tuning:** Learn virtual tokens prepended to input (not demoed here)
- **QLoRA:** LoRA on top of a 4-bit quantized base model -- covered right after this section
  (Concept 7), since it's the natural next question once you've seen how little LoRA already needs

This cell applies LoRA to _continued pretraining_ (not instruction tuning) to show the parameter axis
and data axis are independent choices.


In [ ]:
lora_config_pt = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
)

lora_pt_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
lora_pt_model = get_peft_model(lora_pt_base, lora_config_pt)
lora_pt_model.print_trainable_parameters()

### Building the Dataset

A different 3-genre slice than the other continued-pretraining run, reusing the same
`tokenize_causal()` from Concept 1 -- LoRA changes *how many* parameters train, not *what* they train
on.

In [ ]:
lora_pt_dataset = Dataset.from_dict(
    {
        "text": load_corpus_paragraphs(
            novels=["mystery", "horror", "literary"], max_chapters=3
        )
    }
)
lora_pt_tokenized = lora_pt_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)

### Training and Saving

Same `2e-4` LoRA learning rate as instruction tuning, same `Trainer` pattern as every training cell in
this notebook -- this is the last of the five checkpoints this notebook trains before the head-to-head
comparison further down.

In [ ]:
training_args_lora_pt = TrainingArguments(
    output_dir="./checkpoints/peft-lora",
    per_device_train_batch_size=2,
    max_steps=60,  # gpt2-medium needs more steps than distilgpt2 did to actually settle
    logging_steps=10,
    save_strategy="no",
    learning_rate=2e-4,
    report_to="none",
)

trainer_lora_pt = Trainer(
    model=lora_pt_model, args=training_args_lora_pt, train_dataset=lora_pt_tokenized
)
trainer_lora_pt.train()
lora_pt_model.save_pretrained("./checkpoints/peft-lora")
print("Saved parameter-efficient (LoRA) continued-pretraining adapter.")

### LoRA Decomposition: A Visual Intuition

Let's visualize exactly **what LoRA does** with concrete matrix dimensions. We'll use a tiny example
to make it crystal clear.

**Scenario:** `gpt2-medium`'s attention weight matrix `W_attn` is 1024×1024 (1,048,576 parameters).

**Full fine-tuning** would update: `W_new = W_old + ΔW` where ΔW is also 1024×1024 (1,048,576
trainable params).

**LoRA** instead uses: `W_new = W_old + B·A` where:

- `B` is 1024×8 (8,192 parameters)
- `A` is 8×1024 (8,192 parameters)
- **Total:** 16,384 trainable parameters (1.56% of the original!)

**Key insight:** The rank bottleneck (r=8) forces the update to live in a low-dimensional subspace.
This is like saying "all the adaptation you need can be expressed as 8 basis vectors" instead of the
full 1024-dimensional freedom.

**What does "rank" actually mean, and what is "intrinsic rank"?**

Forget matrices for a second: the **rank** of a matrix is just "how many genuinely independent rows
(or columns) does it have?" A 1024×1024 matrix can be *stored* as over a million numbers, but if every
row is some combination of only 8 truly independent rows, its rank is 8 -- the other million-plus
numbers are redundant, fully determined by those 8. **Low rank** means exactly that: the matrix
*looks* huge, but the information it actually carries fits in a much smaller number of independent
directions. `B(Ax)` is LoRA's way of only ever being *able* to produce rank-8 updates: `A` first
squeezes the input down to 8 numbers, `B` blows those 8 numbers back up to 1024 -- no matter what `A`
and `B` learn, the result can never have more than 8 independent directions in it, by construction.

That's a design choice, not an accident, and it rests on a real empirical finding:
[Aghajanyan et al. 2020](https://arxiv.org/abs/2012.13255) showed that pretrained language models have
a surprisingly low **intrinsic dimension** -- you can fine-tune one for a new task almost as well by
only ever moving within a small random subspace of the full parameter space as you can by updating
every parameter freely. LoRA's authors ([Hu et al. 2021](https://arxiv.org/abs/2106.09685)) extended
that idea specifically to the *weight update* `ΔW`: they hypothesized `ΔW` itself has a low
**intrinsic rank** during adaptation -- the pretrained model already encodes almost everything needed
for the new task, so the *correction* it needs is small and concentrated, not spread across all
1,048,576 numbers `ΔW` could in principle contain. `r=8` isn't picked to save memory first and ask
questions later -- it's a bet, backed by that empirical finding, that 8 independent directions are
*already enough* to capture the real adaptation.

**Why does this work?**

1. **Task adaptations are low-rank:** Fine-tuning for a specific task doesn't need to change every
   direction in the 1024D space — most of the "general language understanding" can stay frozen.
2. **Overfitting resistance:** With fewer parameters, LoRA is less likely to memorize the training
   data and more likely to learn generalizable patterns.
3. **Efficient gradient flow:** The low-rank bottleneck acts as a regularizer, concentrating gradient
   updates into the most important directions.


In [ ]:
# Visualize LoRA matrix decomposition
from matplotlib.patches import Patch, Rectangle

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

d, r = base_model.config.n_embd, 8  # dimension (gpt2-medium: 1024) and rank

# Plot 1: Full fine-tuning ΔW
axes[0].add_patch(Rectangle((0, 0), d, d, fill=True, color="steelblue", alpha=0.6))
axes[0].set_xlim(0, d)
axes[0].set_ylim(0, d)
axes[0].set_aspect("equal")
axes[0].set_title(
    f"Full Fine-Tuning: ΔW\n{d}×{d} = {d*d:,} trainable params", fontsize=11
)
axes[0].set_xlabel(f"{d}")
axes[0].set_ylabel(f"{d}")
axes[0].text(
    d / 2,
    d / 2,
    f"{d*d:,}\nparameters",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color="white",
)
axes[0].legend(
    handles=[Patch(facecolor="steelblue", alpha=0.6, edgecolor="black", label="Full ΔW (dense)")],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    fontsize=8,
)

# Plot 2: LoRA matrix A -- hits the input FIRST, projects d dims DOWN to r dims
axes[1].add_patch(Rectangle((0, 0), d, r, fill=True, color="mediumseagreen", alpha=0.7))
axes[1].set_xlim(0, d)
axes[1].set_ylim(0, r + 100)
axes[1].set_aspect("equal")
axes[1].set_title(
    f"LoRA Matrix A (down-project)\n{r}×{d} = {r*d:,} params", fontsize=11
)
axes[1].set_xlabel(f"{d}")
axes[1].set_ylabel(f"{r}")
axes[1].text(
    d / 2,
    r / 2,
    f"{r*d:,}\nparams",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color="white",
)
axes[1].legend(
    handles=[Patch(facecolor="mediumseagreen", alpha=0.7, edgecolor="black", label="A (down-project, trainable)")],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    fontsize=8,
)

# Plot 3: LoRA matrix B -- takes A's r-dim output and projects it back UP to d dims
axes[2].add_patch(Rectangle((0, 0), r, d, fill=True, color="coral", alpha=0.7))
axes[2].set_xlim(0, r + 100)
axes[2].set_ylim(0, d)
axes[2].set_aspect("equal")
axes[2].set_title(f"LoRA Matrix B (up-project)\n{d}×{r} = {d*r:,} params", fontsize=11)
axes[2].set_xlabel(f"{r}")
axes[2].set_ylabel(f"{d}")
axes[2].text(
    r / 2,
    d / 2,
    f"{d*r:,}\nparams",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color="white",
)
axes[2].legend(
    handles=[Patch(facecolor="coral", alpha=0.7, edgecolor="black", label="B (up-project, trainable)")],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    fontsize=8,
)

# Plot 4: B(Ax) result (low-rank approximation)
axes[3].add_patch(Rectangle((0, 0), d, d, fill=True, color="mediumpurple", alpha=0.6))
axes[3].set_xlim(0, d)
axes[3].set_ylim(0, d)
axes[3].set_aspect("equal")
axes[3].set_title(
    f"B(Ax) = Low-Rank ΔW\n{d}×{d} with rank {r}\nTotal: {d*r + r*d:,} params",
    fontsize=11,
)
axes[3].set_xlabel(f"{d}")
axes[3].set_ylabel(f"{d}")
axes[3].text(
    d / 2,
    d / 2,
    f"rank-{r}\nupdate",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color="white",
)
axes[3].annotate(
    "",
    xy=(d / 2, d - 50),
    xytext=(d / 2, 50),
    arrowprops=dict(arrowstyle="<->", lw=2, color="yellow"),
)
axes[3].text(
    d / 2 + 80,
    d / 2,
    f"Only {r} degrees\nof freedom!",
    fontsize=9,
    color="yellow",
    fontweight="bold",
)
axes[3].legend(
    handles=[Patch(facecolor="mediumpurple", alpha=0.6, edgecolor="black", label="B(A(x)) low-rank ΔW")],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    fontsize=8,
)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

# Print parameter savings
full_params = d * d
lora_params = d * r + r * d
saving_pct = (1 - lora_params / full_params) * 100

print(f"\n{'=' * 70}")
print(f"Parameter Efficiency Analysis (d={d}, r={r}):")
print(f"{'=' * 70}")
print(f"  Full fine-tuning:  {full_params:>10,} parameters (100.0%)")
print(
    f"  LoRA adaptation:   {lora_params:>10,} parameters ({lora_params/full_params*100:>5.2f}%)"
)
print(
    f"  Savings:           {full_params - lora_params:>10,} parameters ({saving_pct:>5.2f}%)"
)
print(f"{'=' * 70}")
print(f"  Memory savings: ~{saving_pct:.1f}% less optimizer state")
print(f"  Training speed: ~{saving_pct:.1f}% fewer gradients to compute")
print(f"  Swappability: Can load/unload adapters in <1 second")
print(f"{'=' * 70}")

### Cracking Open a Real LoRA-Adapted Model

The diagram above is the toy math (`d x d`, `r=8`, hand-picked numbers). Let's now open
`instruct_lora_model` -- the real LoRA adapter we already trained earlier for instruction tuning --
and look at the actual PyTorch modules PEFT injected, then trace one real forward pass through an
adapted layer to see the frozen base output and the tiny LoRA delta side by side, on real activations
instead of shapes.


In [ ]:
# Find every module PEFT actually injected LoRA adapters into
lora_layers = [
    (name, module)
    for name, module in instruct_lora_model.named_modules()
    if hasattr(module, "lora_A") and len(getattr(module, "lora_A")) > 0
]
print(
    f"PEFT wrapped {len(lora_layers)} layers with LoRA adapters "
    f"(one c_attn per transformer block)."
)
print("First 3 wrapped module names:")
for name, _ in lora_layers[:3]:
    print(" ", name)

# Zoom into the very first block's adapted attention projection
name0, layer0 = lora_layers[0]
base0 = layer0.base_layer
lora_A0 = layer0.lora_A["default"]
lora_B0 = layer0.lora_B["default"]
scaling0 = layer0.scaling["default"]

print(f"\nInside {name0}:")
print(
    f"  Frozen base layer:   {type(base0).__name__}, weight shape {tuple(base0.weight.shape)}"
)
print(f"  lora_A (down-proj):  {tuple(lora_A0.weight.shape)}  <- trainable")
print(f"  lora_B (up-proj):    {tuple(lora_B0.weight.shape)}  <- trainable")
print(f"  scaling (alpha/r):   {scaling0}")
print(
    f"  Trained lora_B norm: {lora_B0.weight.norm().item():.4f}  "
    f"(this was ~0.0 before training -- lora_B is zero-initialized so the "
    f"adapter starts as a no-op)"
)

### Verifying the Rank Claim on the Real Trained Adapter

`lora_A0`/`lora_B0` above are real, trained matrices -- so the "rank ≤ r" claim from the intuition
section isn't just a shape argument, it's checkable. Build the actual effective update this adapter
contributes, `ΔW = scaling · B·A`, and look at its singular values: by construction, no more than `r`
of them can be non-zero, no matter what values training found for `A` and `B`. This is the real
mechanics behind "low-rank update," measured on the exact adapter trained earlier in this notebook --
not a toy shape diagram.


In [ ]:

# Verify LoRA's rank constraint on the real trained adapter -- not asserted, measured via SVD.
delta_W = layer0.get_delta_weight("default").detach().cpu()  # PEFT's own scaling * B @ A computation
singular_values = torch.linalg.svdvals(delta_W)
r = lora_A0.weight.shape[0]  # the configured rank, read directly off the trained A matrix

# Only the first few dozen singular values are ever non-negligible for a rank-r update -- plotting
# all min(delta_W.shape) of them would squeeze the real cliff into an invisible sliver, so zoom in
# and use a log y-axis, which makes an 8-orders-of-magnitude drop actually visible.
n_show = min(30, len(singular_values))
floor = 1e-8  # log scale needs a positive floor; true near-zero values are clipped up for display only
plot_values = np.clip(singular_values[:n_show].numpy(), floor, None)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(
    range(n_show),
    plot_values,
    color=["mediumseagreen" if i < r else "lightgray" for i in range(n_show)],
)
ax.set_yscale("log")
ax.set_xlabel(f"Singular value index (first {n_show} of {len(singular_values)})")
ax.set_ylabel("Singular value magnitude (log scale)")
ax.set_title(
    f"Singular Value Spectrum of the Real Trained \u0394W = scaling \u00b7 B\u00b7A\n"
    f"({tuple(delta_W.shape)} matrix, configured rank r={r})",
    fontsize=11,
    fontweight="bold",
)
ax.legend(
    handles=[
        Patch(facecolor="mediumseagreen", edgecolor="black",
              label=f"First {r} singular values (the adapter's real degrees of freedom)"),
        Patch(facecolor="lightgray", edgecolor="black",
              label=f"Indices {r + 1}-{n_show} (~0 by construction, floored for the log scale)"),
    ],
    fontsize=8,
)
plt.tight_layout()
plt.show()

nonzero = (singular_values > 1e-4).sum().item()
print(f"\u0394W shape: {tuple(delta_W.shape)} -> full rank would allow up to {min(delta_W.shape)} singular values")
print(f"Singular values above 1e-4: {nonzero} (matches the configured rank r={r})")
print(f"Largest singular value: {singular_values[0].item():.4f}   {r}th singular value: {singular_values[r - 1].item():.4f}")
print(f"First value past the rank cutoff (index {r}): {singular_values[r].item():.2e}  <- effectively zero")
print(
    "This is the real mechanics behind 'low-rank update': it isn't that training happened to find a "
    "low-rank \u0394W -- B(A(x))'s construction makes it mathematically impossible for \u0394W to have "
    "more than r independent directions, no matter what A and B learn."
)


### Tracing a Real Forward Pass Through the Adapted Layer

`lora_B` started at all-zeros, so before training this adapter was a mathematical no-op: the combined
output was _exactly_ the frozen base output. After training, `lora_B` has real values (norm printed
above), so now it nudges the output. Let's actually capture both paths -- the frozen base output and
the full (base + LoRA) output -- for a real prompt, using forward hooks (no reimplementing the math by
hand, so there's no risk of getting the internal computation subtly wrong).


In [ ]:
# Capture the frozen base output and the combined (base + LoRA) output for a real prompt,
# using forward hooks -- this measures what PEFT actually computes, not a re-derivation of it.
from matplotlib.patches import Patch

captured = {}


def make_hook(key):
    def hook(module, inputs, output):
        captured[key] = output.detach()

    return hook


hook_base = base0.register_forward_hook(make_hook("base_only"))
hook_combined = layer0.register_forward_hook(make_hook("combined"))

demo_prompt = INSTRUCTION_PREFIX + "Aria Voss checked the Meridian's Promise and"
enc_lora = tokenizer(demo_prompt, return_tensors="pt").to(device)

instruct_lora_model.eval()
with torch.no_grad():
    _ = instruct_lora_model(**enc_lora)

hook_base.remove()
hook_combined.remove()

base_out    = captured["base_only"][0]   # (seq_len, 3072) — frozen W0 @ x + bias
combined_out = captured["combined"][0]   # (seq_len, 3072) — base_out + scaling * B(A(x))
lora_delta  = combined_out - base_out    # isolate just the adapter's real contribution

last_pos = base_out.shape[0] - 1        # last token position carries the richest context
show_dims = 60                           # first 60 of 3072 c_attn dimensions

print(f"Prompt: {demo_prompt!r}")
print(f"c_attn output dim: {base_out.shape[-1]} (= 3 × n_embd, packed Q/K/V for gpt2-medium)")
print(f"\nAt the last token position:")
print(f"  ||base output||   = {base_out[last_pos].norm().item():.3f}")
print(f"  ||LoRA delta||    = {lora_delta[last_pos].norm().item():.5f}")
print(
    f"  delta / base norm = {(lora_delta[last_pos].norm() / base_out[last_pos].norm()).item():.4%}"
    " ← the adapter nudges the output by a small fraction, it doesn't replace it"
)

# ── Static panel: Base vs Combined + LoRA Delta at last position ─────────────
fig_static, (ax_static1, ax_static2) = plt.subplots(1, 2, figsize=(14, 4))

ax_static1.plot(base_out[last_pos, :show_dims].numpy(),
                color="steelblue", label="base (frozen W0 × x)")
ax_static1.plot(combined_out[last_pos, :show_dims].numpy(),
                color="coral", linestyle="--", label="combined (base + LoRA)")
ax_static1.set_title("Base vs. Combined Output\n(first 60 of 3072 c_attn dims)", fontsize=11)
ax_static1.set_xlabel("Output dimension")
ax_static1.legend(fontsize=8)
ax_static1.grid(alpha=0.3)

ax_static2.bar(np.arange(show_dims), lora_delta[last_pos, :show_dims].numpy(),
               color="mediumseagreen", label="LoRA delta (combined - base)")
ax_static2.set_title(
    f"LoRA Delta at token position {last_pos} (the last token)\n"
    "What lora_B(lora_A(x)) × scaling actually adds", fontsize=11)
ax_static2.set_xlabel("Output dimension")
ax_static2.axhline(0, color="black", linewidth=0.8)
ax_static2.legend(fontsize=8)
ax_static2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Animated panel: watch the LoRA delta evolve token by token ───────────────
# Each animation frame reveals the delta bar-chart for one more token position,
# so you can see how the adapter's correction changes as the model builds context.
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_tokens  = lora_delta.shape[0]
delta_arr = lora_delta[:, :show_dims].numpy()
delta_max = np.abs(delta_arr).max() * 1.15 or 1e-6   # graceful fallback if all zeros

fig_anim, ax_anim = plt.subplots(figsize=(12, 4))
fig_anim.patch.set_facecolor("#f8f8f8")
ax_anim.set_facecolor("#f8f8f8")

bar_colors  = ["mediumseagreen" if v >= 0 else "coral" for v in delta_arr[0]]
bars_anim   = ax_anim.bar(np.arange(show_dims), delta_arr[0], color=bar_colors)
ax_anim.axhline(0, color="black", linewidth=0.8)
ax_anim.set_ylim(-delta_max, delta_max)
ax_anim.set_xlim(-1, show_dims)
ax_anim.set_xlabel("Output dimension (first 60 of 3072)", fontsize=10)
ax_anim.set_ylabel("LoRA delta value", fontsize=10)
ax_anim.grid(alpha=0.3)
ax_anim.legend(
    handles=[
        Patch(facecolor="mediumseagreen", label="Positive delta (boosts dimension)"),
        Patch(facecolor="coral", label="Negative delta (suppresses dimension)"),
    ],
    loc="upper right",
    fontsize=8,
)

# Show the decoded token at each position for context
tokens_decoded = tokenizer.convert_ids_to_tokens(enc_lora["input_ids"][0].tolist())

title_obj = ax_anim.set_title("", fontsize=11, fontweight="bold")

def _update(frame):
    deltas = delta_arr[frame]
    for bar, d in zip(bars_anim, deltas):
        bar.set_height(d)
        bar.set_color("mediumseagreen" if d >= 0 else "coral")
    tok = tokens_decoded[frame] if frame < len(tokens_decoded) else "?"
    title_obj.set_text(
        f"LoRA Delta — token {frame}/{n_tokens - 1}  '{tok}'\n"
        f"||delta|| = {np.linalg.norm(deltas):.5f}"
    )
    return list(bars_anim) + [title_obj]

anim = FuncAnimation(fig_anim, _update, frames=n_tokens, interval=160, blit=False)
plt.close(fig_anim)    # suppress the static inline display; the HTML widget takes over

print(
    f"\nAnimation: {n_tokens} frames, one per token in the prompt.\n"
    "Green bars = positive delta (adapter boosts this c_attn dimension).\n"
    "Red bars   = negative delta (adapter suppresses it).\n"
    "Watch the pattern shift as context accumulates — early tokens show almost no signal,\n"
    "late tokens (the last few) carry the richest context so the adapter fires harder.\n"
)
display(HTML(anim.to_jshtml(fps=6)))

print(
    "\nThis is the whole story: the frozen Conv1D still does all the heavy lifting (base output), "
    "and the tiny rank-8 adapter adds a small, learned correction on top — computed fresh for "
    "every token position, every forward pass."
)


### Adapter Portability: Swapping Onto a Different Base

The LoRA Pros list above claimed adapters are **swappable** — one frozen base, multiple adapters
loaded per request. Let's test it rather than take it on faith.

**What "different base" actually means:** an adapter encodes the low-rank deltas in terms of the
layer shapes it was trained against. It doesn't embed any actual weights from the original checkpoint.
That means the adapter file will load onto *any* model whose target-module weight shapes match —
including:

- A fresh instantiation of `gpt2-medium` from HuggingFace (new Python object, same shapes)
- A `gpt2-medium` that was itself independently fine-tuned or even partially quantized (QLoRA, next section)
- Any future checkpoint that keeps the same hidden dimension and target modules

What it **cannot** do: load onto a different architecture where the shapes don't match (e.g., `gpt2`
at d=768 vs `gpt2-medium` at d=1024 — PEFT will raise a shape error immediately).

The code below loads the `peft-lora` adapter onto a freshly instantiated base, then **swaps in the
`instruction-lora` adapter from Part 1 on the same base object** without reloading any weights — the
swap is just a pointer change inside PEFT's routing layer.


In [ ]:
import gc, os
from peft import PeftModel

# --- Adapter portability demo ---

# 1. Load a fresh gpt2-medium base (new Python object, weights come from HuggingFace cache --
#    no re-download needed, but this is fully independent of base_model and lora_pt_model)
swap_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

# 2. Attach the peft-lora adapter (corpus pretraining, saved in Concept 6)
swap_model = PeftModel.from_pretrained(swap_base, "./checkpoints/peft-lora")
swap_model.eval()

adapter_kb = sum(
    os.path.getsize(os.path.join("./checkpoints/peft-lora", f))
    for f in os.listdir("./checkpoints/peft-lora")
) / 1024
base_mb = sum(p.numel() * p.element_size() for p in swap_base.parameters()) / 1024 / 1024
print(f"Base model in memory : {base_mb:.0f} MB  (shared — never reloaded per adapter)")
print(f"peft-lora on disk    : {adapter_kb:.0f} KB  (the only per-task file)")

print(f"\n[Adapter 1 — peft-lora] continuation:")
print(f"  {generate(swap_model, PROMPT)}")

# 3. Swap in the instruction-lora adapter from Part 1 — same base object, no weights reloaded
swap_model.load_adapter("./checkpoints/instruction-lora", adapter_name="instruct")
swap_model.set_adapter("instruct")

instruct_kb = sum(
    os.path.getsize(os.path.join("./checkpoints/instruction-lora", f))
    for f in os.listdir("./checkpoints/instruction-lora")
) / 1024
print(f"\n[Adapter 2 — instruction-lora, swapped in] adapter size: {instruct_kb:.0f} KB")
print(f"  {generate(swap_model, INSTRUCTION_PREFIX + PROMPT + chr(10) * 2)}")

# 4. Switch back to the first adapter — instant, just a dict key change in PEFT's routing
swap_model.set_adapter("default")
print(f"\n[Adapter 1 — peft-lora, switched back]")
print(f"  {generate(swap_model, PROMPT)}")

# 5. Show what happens with a mismatched architecture (just inspect, no full load needed)
print("\nShape check: would this adapter load onto gpt2 (d=768)?")
peft_lora_A_shape = tuple(swap_model.base_model.model.transformer.h[0].attn.c_attn.lora_A["default"].weight.shape)
print(f"  peft-lora A matrix shape: {peft_lora_A_shape}  (r=8, d={peft_lora_A_shape[1]})")
print(f"  gpt2's c_attn input dim : 768  ← shape mismatch (adapter expects d=1024) → PEFT raises an error at load time")
print(f"  Same adapter on gpt2-large (d=1280) would also fail for the same reason")
print(f"  Works on: any gpt2-medium checkpoint with the same hidden dim 1024")

del swap_base, swap_model
gc.collect()

### Concept 7 (Parameter-Based): QLoRA - LoRA on a Quantized Base

**Riverside's question for this section:** LoRA already reduced trainable parameters to well under
1%. But the *frozen* base model itself is still ~355M parameters at full precision, sitting in memory
the whole time. What if the base model itself were bigger -- Riverside's next hardware refresh, or a
jump to a 7B/13B model -- and even the frozen copy didn't fit comfortably? Is there a way to shrink
that too, without touching the LoRA idea itself?

**The distinction LoRA alone doesn't make:** LoRA already answered "how many *trainable* parameters
do we need" (answer: very few). It never touched a separate question: "how much memory does the
*frozen, unchanging* copy of the base model itself take up?" For `gpt2-medium` on a laptop, ~710MB of
frozen fp32 weights is a rounding error. For a 65B-parameter model, the frozen base alone is ~130GB in
fp16 -- more memory than any single consumer GPU has, LoRA adapter or not. QLoRA
([Dettmers et al. 2023](https://arxiv.org/abs/2305.14314)) closes exactly that gap.

**What it is:** Take the ordinary LoRA recipe from Concept 6 -- freeze the base, train small adapter
matrices -- and additionally **quantize the frozen base weights themselves down to 4 bits** before
ever starting to train. Three ideas from the QLoRA paper make this actually work:

1. **NF4 (4-bit NormalFloat):** a quantization data type designed for weights that are (empirically,
   for pretrained neural nets) close to normally distributed -- it spaces its 16 representable values
   so each one covers an equal amount of *probability mass* under a standard normal, not an equal
   numeric range like a naive linear int4 scheme would. That match to the actual weight distribution
   is what keeps a 4-bit base model's quality close to its 16-bit original.
2. **Double quantization:** even the small per-block scaling constants that quantization itself needs
   are quantized a second time, shaving off a further ~0.4 bits/parameter on average -- a
   detail-level optimization, but one of the paper's real, measured savings.
3. **Paged optimizers:** borrow NVIDIA's unified memory to automatically page optimizer state between
   GPU and CPU RAM during rare, sudden memory spikes (e.g., an unusually long sequence in a batch), so
   training doesn't crash instead of just slowing down for that one step.

The result: the frozen base model's memory footprint drops ~4x (16-bit -> 4-bit) *on top of* whatever
the LoRA adapter's own trainable-parameter savings already bought -- which is what let the QLoRA paper
fine-tune a 65B-parameter model on a single 48GB consumer GPU, a model that would need multiple 80GB
A100s in fp16.

**Pros:**

- Combines LoRA's tiny trainable-parameter footprint with a ~4x smaller frozen base -- the two
  savings stack
- Enables fine-tuning models far larger than the same GPU could handle even with ordinary LoRA
- NF4 is specifically tuned to pretrained weight distributions, so quality loss vs. full-precision
  LoRA is small in practice (the paper reports parity with 16-bit LoRA on their benchmarks)

**Cons:**

- Needs `bitsandbytes`' custom CUDA kernels for the 4-bit dequantize-on-the-fly forward/backward
  pass -- **GPU-specific**, no CPU path exists
- Slightly slower per-step than full-precision LoRA (the extra dequantization work), though this is
  usually dwarfed by the memory savings that make training possible at all
- One more moving part to configure correctly (quantization config + LoRA config together)

**Why this notebook doesn't run it:** Riverside's whole premise is a laptop CPU, and `bitsandbytes`'
4-bit kernels require CUDA -- there is no CPU fallback, unlike ordinary LoRA (which is just matrix
multiplication, CPU-friendly by construction). This is the same honest line Part 1 drew around PPO in
the DPO section: explained precisely, contrasted with what actually got trained, not run. The code
block below is what you'd actually write on a GPU machine -- real, working configuration, just not
executed here.

```python
# Illustrative only -- requires a CUDA GPU + bitsandbytes. Not executed in this CPU notebook.
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # the NormalFloat4 scheme described above
    bnb_4bit_use_double_quant=True,         # quantize the quantization constants too
    bnb_4bit_compute_dtype=torch.bfloat16,  # de-quantize to bf16 for the actual matmul
)

quantized_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
quantized_base = prepare_model_for_kbit_training(quantized_base)  # grad checkpointing, fp32 norms

qlora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16,
    target_modules=["c_attn"], lora_dropout=0.05, bias="none",
)
qlora_model = get_peft_model(quantized_base, qlora_config)
qlora_model.print_trainable_parameters()  # same tiny trainable count as ordinary LoRA --
                                           # the difference is entirely in how the frozen base is stored
```

The `LoraConfig` here is *identical* to Concept 6's -- QLoRA doesn't change what LoRA trains, only
what the frozen base weights are stored as. That's the cleanest way to keep the two ideas straight:
**LoRA is a training-time trainable-parameter-count decision; quantization is a storage/precision
decision that applies to the frozen base, independent of LoRA.** QLoRA is just both decisions made at
once.


### Quantization for Deployment: Shrinking the Model Riverside Actually Ships

QLoRA above uses quantization to make *training* fit in less memory. That's a different question from
the one Riverside will actually face once a checkpoint is picked and ready to deploy: **can the
deployed model itself be made smaller and faster to run, independent of how it was trained?** This is
a real, separate decision -- you can quantize a model that was never trained with LoRA at all, and you
can serve a LoRA/QLoRA-trained model at full precision if you want.

**The idea, in one sentence:** replace each weight's 32-bit floating point number with a
lower-precision representation (commonly 8-bit or 4-bit integers, plus a small per-tensor or
per-channel scale factor to convert back), trading a small, usually-imperceptible amount of numerical
precision for a large reduction in memory and, often, faster inference.

| Technique                                              | What's quantized                                                            | Calibration data needed?           | Typical bit-width       | Hardware                    | Notes |
| -------------------------------------------------------- | ----------------------------------------------------------------------------- | ------------------------------------ | ------------------------- | ----------------------------- | ----- |
| **Dynamic quantization** (PyTorch native)                | Weights ahead of time; activations on-the-fly at inference                    | No                                    | int8                       | **CPU only**                  | The only technique here that's genuinely CPU-native with zero extra setup -- what the code cell below actually runs |
| **Static / post-training quantization (PTQ)**            | Weights and activations, both ahead of time                                    | Yes -- a representative calibration batch | int8                       | CPU/GPU                       | Faster than dynamic (no on-the-fly work) but brittle if the calibration set doesn't match real traffic |
| **Quantization-aware training (QAT)**                    | Weights and activations, simulated during training                             | Full retraining pass                 | int8 (sometimes lower)     | CPU/GPU                       | Best accuracy retention of the int8 options, since the model learns to compensate for quantization noise -- but it's a training job, not a five-minute conversion |
| **GPTQ**                                                 | Weights, via layer-by-layer reconstruction against a small calibration set      | Yes, small (~128 samples)            | int4 (commonly)            | GPU (fast custom kernels)     | Popular for shipping large open-weight LLMs at 4-bit; one-shot, doesn't need the original training data |
| **AWQ** (Activation-aware Weight Quantization)           | Weights, with a per-channel scale chosen to protect the ~1% of weight channels activations rely on most | Yes, small calibration set            | int4 (commonly)            | GPU                           | Often matches or beats GPTQ quality at similar speed, by explicitly *not* punishing the channels that matter most |
| **bitsandbytes NF4** (QLoRA's scheme, Concept 7 above)   | Weights, using the NormalFloat4 data type                                       | No (data-type change, not calibrated) | 4-bit                       | GPU only                      | Built for *training* through a quantized base, not primarily a deployment-time compression format, though the same 4-bit weights can also be used purely for inference |

**Which one is Riverside's laptop actually able to run?** Only dynamic quantization -- it needs no
calibration dataset, no GPU, and no retraining, which is exactly why it's the one technique below with
a real, executed code cell instead of just a description. GPTQ, AWQ, static PTQ, and QAT are all
genuinely valuable in production (especially for serving very large models cheaply), but they either
require GPU kernels this laptop doesn't have, a calibration pipeline this notebook doesn't build, or a
full retraining pass -- named and compared honestly above, not implemented, the same treatment this
notebook already gave PPO and QLoRA's own NF4 kernels.


In [ ]:
# Real, CPU-native post-training quantization -- the only technique in the table above we can
# actually run without a GPU or a calibration pipeline. Quantize Part 1's continued-pretraining
# checkpoint and measure the real memory and quality cost.
#
# Deliberately pinned to CPU regardless of `device`: PyTorch's dynamic quantization only supports
# CPU inference -- which happens to be a perfect match for Riverside's deployment target, unlike
# QLoRA above (no GPU/CPU tug-of-war to work around here).
import io
import math

fp32_model = AutoModelForCausalLM.from_pretrained("./checkpoints/non-instruction-full")
fp32_model.eval()

# Every nn.Linear layer's weights become int8; activations are quantized on-the-fly per batch at
# inference time (hence "dynamic").
int8_model = torch.quantization.quantize_dynamic(
    fp32_model, {torch.nn.Linear}, dtype=torch.qint8
)


def state_dict_size_mb(model):
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.getbuffer().nbytes / 1e6


fp32_mb = state_dict_size_mb(fp32_model)
int8_mb = state_dict_size_mb(int8_model)
print(f"fp32 state_dict size: {fp32_mb:.1f} MB")
print(f"int8 state_dict size: {int8_mb:.1f} MB")
print(f"Real size reduction:  {(1 - int8_mb / fp32_mb) * 100:.1f}%")


def load_tail_paragraphs(novels, start_chapter_index=10, chapters_per_novel=2, min_len=200):
    """Paragraphs from later chapters -- guaranteed unseen by every training run in this arc, the
    same held-out idea Part 3 uses at full scale, kept small and local here for a quick check."""
    paragraphs = []
    for alias in novels:
        novel_path = CONTENT_DIR / NOVELS[alias]
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))
        for path in chapter_files[start_chapter_index : start_chapter_index + chapters_per_novel]:
            text = path.read_text(encoding="utf-8")
            for para in text.split("\n\n"):
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:
                    paragraphs.append(para)
    return paragraphs


def quick_perplexity(model, paragraphs, max_length=128):
    model.eval()
    losses = []
    with torch.no_grad():
        for para in paragraphs:
            enc = tokenizer(para, truncation=True, max_length=max_length, return_tensors="pt")
            out = model(**enc, labels=enc["input_ids"])
            losses.append(out.loss.item())
    return math.exp(sum(losses) / len(losses))


def generate_cpu(model, prompt, max_new_tokens=40):
    """Same idea as the shared generate() helper, but deliberately never calls .to(device) --
    fp32_model/int8_model above are plain CPU tensors, so this avoids a device mismatch if
    `device` happens to be "cuda" elsewhere in this notebook."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()


quant_holdout = load_tail_paragraphs(["scifi", "fantasy"])
fp32_ppl = quick_perplexity(fp32_model, quant_holdout)
int8_ppl = quick_perplexity(int8_model, quant_holdout)
print(f"\nHeld-out perplexity, fp32: {fp32_ppl:.1f}")
print(f"Held-out perplexity, int8: {int8_ppl:.1f}  (real cost of quantizing this checkpoint)")

print("\n=== Same prompt, both precisions ===")
print(f"fp32: {generate_cpu(fp32_model, PROMPT)}")
print(f"int8: {generate_cpu(int8_model, PROMPT)}")

print(
    "\nRiverside's takeaway: dynamic quantization is a real, ~free lever to pull at deployment "
    "time -- a meaningfully smaller state_dict for a perplexity cost that (if the numbers above "
    "came out close) is easy to accept, independent of whether the checkpoint was trained with "
    "full fine-tuning, partial freezing, LoRA, or QLoRA in the first place."
)


### Visual Comparison: Parameter Counts Across All Techniques

Before diving into the LoRA code, let's visualize **exactly how much memory/compute each parameter-
based approach requires**. This builds the intuition for why LoRA has become the industry standard.

> **Where's QLoRA on this chart?** Nowhere -- deliberately. QLoRA's *trainable*-parameter count would
> land at roughly the same tiny bar as LoRA (same adapter, same rank); what it additionally saves is
> frozen *base-weight* memory, a dimension this chart doesn't plot, and it was never trained in this
> run (GPU-specific, per Concept 7). Adding a bar for a number we didn't measure would be exactly the
> kind of fabricated chart this notebook avoids elsewhere.



In [ ]:
# Visual parameter comparison across all techniques
from matplotlib.patches import FancyBboxPatch, Patch

# Real parameter counts pulled from the actual models trained earlier in this notebook
# (not hardcoded, so this stays correct no matter which base model MODEL_NAME points to)
total_params = sum(p.numel() for p in base_model.parameters())
full_ft_params = total_params  # 100%
partial_ft_params = sum(p.numel() for p in freeze_model.parameters() if p.requires_grad)
lora_params = sum(
    p.numel() for p in lora_pt_model.parameters() if p.requires_grad
)

techniques = ["Full\nFine-Tuning", "Partial\nFreezing", "LoRA"]
param_counts = [full_ft_params, partial_ft_params, lora_params]
param_pcts = [count / total_params * 100 for count in param_counts]
colors = ["steelblue", "coral", "mediumseagreen"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Absolute parameter counts (log scale for visibility)
bars = ax1.bar(
    techniques, param_counts, color=colors, alpha=0.8, edgecolor="black", linewidth=1.5
)
ax1.set_yscale("log")
ax1.set_ylabel("Trainable Parameters (log scale)", fontsize=12, fontweight="bold")
ax1.set_title(
    f"Absolute Parameter Counts ({MODEL_NAME})", fontsize=13, fontweight="bold"
)
ax1.grid(alpha=0.3, axis="y")
ax1.legend(
    handles=[
        Patch(facecolor=c, alpha=0.8, edgecolor="black", label=t.replace("\n", " "))
        for t, c in zip(techniques, colors)
    ],
    loc="upper right",
    fontsize=9,
)

# Annotate bars with exact counts
for i, (bar, count) in enumerate(zip(bars, param_counts)):
    height = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width() / 2.0,
        height,
        f"{count:,}\n({param_pcts[i]:.2f}%)",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

# Plot 2: Memory requirements visualization (relative sizes), using the same real numbers
bytes_per_param = 4 + 8  # fp32 weight (4 bytes) + Adam optimizer state (8 bytes)
mem_gb = [count * bytes_per_param / 1e9 for count in param_counts]

ax2.set_xlim(0, 10)
ax2.set_ylim(0, 8)
ax2.axis("off")
ax2.set_title(
    "Relative Memory Footprint\n(trainable params + optimizer state)",
    fontsize=13,
    fontweight="bold",
)

# Full fine-tuning: large box
full_box = FancyBboxPatch(
    (0.5, 4.5),
    9,
    3,
    boxstyle="round,pad=0.1",
    edgecolor="steelblue",
    facecolor="steelblue",
    alpha=0.6,
    linewidth=2,
)
ax2.add_patch(full_box)
ax2.text(
    5,
    6,
    f"Full Fine-Tuning\n{full_ft_params / 1e6:.0f}M params\n~{mem_gb[0]:.2f} GB memory",
    ha="center",
    va="center",
    fontsize=11,
    fontweight="bold",
    color="white",
)

# Partial freezing: medium box
partial_box = FancyBboxPatch(
    (0.5, 2.5),
    6,
    1.5,
    boxstyle="round,pad=0.1",
    edgecolor="coral",
    facecolor="coral",
    alpha=0.7,
    linewidth=2,
)
ax2.add_patch(partial_box)
ax2.text(
    3.5,
    3.25,
    f"Partial Freezing\n{partial_ft_params / 1e6:.0f}M params\n~{mem_gb[1] * 1000:.0f} MB",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color="white",
)

# LoRA: tiny box
lora_box = FancyBboxPatch(
    (0.5, 0.5),
    2,
    1.5,
    boxstyle="round,pad=0.1",
    edgecolor="mediumseagreen",
    facecolor="mediumseagreen",
    alpha=0.8,
    linewidth=2,
)
ax2.add_patch(lora_box)
ax2.text(
    1.5,
    1.25,
    f"LoRA\n{lora_params / 1e3:.0f}K params\n~{mem_gb[2] * 1000:.1f} MB",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color="white",
)

# Add memory savings annotations -- explicitly "vs. full fine-tuning" so the comparison baseline
# is unambiguous (both arrows point back at the Full Fine-Tuning box).
partial_savings_pct = (1 - partial_ft_params / total_params) * 100
lora_savings_pct = (1 - lora_params / total_params) * 100

ax2.annotate(
    "", xy=(7, 5), xytext=(3, 3.5), arrowprops=dict(arrowstyle="->", lw=2, color="red")
)
ax2.text(
    5.5,
    4.5,
    f"{partial_savings_pct:.0f}% less\nthan full FT",
    fontsize=9,
    color="red",
    fontweight="bold",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
)

ax2.annotate(
    "",
    xy=(5, 5.5),
    xytext=(1.5, 2),
    arrowprops=dict(arrowstyle="->", lw=2, color="green"),
)
ax2.text(
    2.5,
    2.5,
    f"{lora_savings_pct:.1f}% less\nthan full FT!",
    fontsize=9,
    color="green",
    fontweight="bold",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
)
ax2.legend(
    handles=[
        Patch(facecolor="steelblue", alpha=0.6, edgecolor="black", label="Full Fine-Tuning (100% params)"),
        Patch(facecolor="coral", alpha=0.7, edgecolor="black", label="Partial Freezing (~10-30% params)"),
        Patch(facecolor="mediumseagreen", alpha=0.8, edgecolor="black", label="LoRA (<1% params)"),
    ],
    loc="lower right",
    fontsize=8,
)

plt.tight_layout()
plt.show()

# Print detailed breakdown
print(f"\n{'=' * 80}")
print(f"Memory & Compute Analysis for {MODEL_NAME} ({total_params / 1e6:.0f}M params):")
print(f"{'=' * 80}")
print(
    f"{'Technique':<20} {'Trainable':<15} {'%':<8} {'Memory Est.':<15} {'Training Speed'}"
)
print(f"{'-' * 80}")
print(
    f"{'Full Fine-Tuning':<20} {f'{full_ft_params:,}':<15} {f'{param_pcts[0]:.2f}%':<8} {f'~{mem_gb[0]:.2f} GB':<15} {'1.0x (baseline)'}"
)
print(
    f"{'Partial Freezing':<20} {f'{partial_ft_params:,}':<15} {f'{param_pcts[1]:.2f}%':<8} {f'~{mem_gb[1] * 1000:.0f} MB':<15} {f'~{total_params / max(partial_ft_params, 1):.1f}x faster'}"
)
print(
    f"{'LoRA (r=8)':<20} {f'{lora_params:,}':<15} {f'{param_pcts[2]:.2f}%':<8} {f'~{mem_gb[2] * 1000:.1f} MB':<15} {f'~{total_params / max(lora_params, 1):.0f}x faster'}"
)
print(f"{'-' * 80}")
print()
print("For a 70B model, these same ratios apply and the differences become MASSIVE:")
print(f"  • Full FT: ~840 GB → requires 8×A100 80GB GPUs")
print(f"  • LoRA:    ~3 GB → fits on a single consumer GPU (RTX 4090)")
print(f"{'=' * 80}")

---

## End of Part 2: Six Checkpoints, One Open Question

Two more checkpoints now exist on disk, joining Part 1's three:

| Checkpoint on disk              | What it is                                                      |
| --------------------------------- | ------------------------------------------------------------------ |
| `./checkpoints/partial-freeze`    | Partial fine-tuning / layer freezing, continued pretraining (Concept 5) |
| `./checkpoints/peft-lora`         | LoRA, continued pretraining (Concept 6)                            |

Between Part 1 and Part 2, Riverside now has **six trained checkpoints** covering five of the nine
data × parameter combinations, plus a real, measured look at post-training quantization and an honest
account of why QLoRA wasn't trained here (GPU-specific). What's still missing: **a real, side-by-side,
quantitative comparison of all six** -- which one actually deserves to be deployed?

Continue to **[Part 3: Comparison & Decision](03-llm-finetuning-comparison-and-decision.ipynb)**, which
reloads all six checkpoints from disk and puts them head-to-head: qualitative side-by-side generations,
token-probability analysis, held-out perplexity, the full data × parameter combination grid, an
ablation study, and the final call on what Riverside House actually ships.
